# CASO 2. CONTROL DE CALIDAD EN UNA PLANTA DE PRODUCCIÓN

**Universidad del Pacífico — Programa de Ingeniería de Sistemas**
**Asignatura: Inteligencia Artificial — Taller Práctico No. 4**
**Librerías: NumPy, Matplotlib y Seaborn**

## 👥 Integrantes

| Nombre |
|--------|
| Jefferson Manuel Valencia Riascos |
| Isnildo Equia Perteaga |
| Sebastian Rojas Cabrera |
| Yeison Stiven Lozano Angulo |

---

## 1. Contexto

Una empresa manufacturera produce piezas metálicas y desea analizar los factores que influyen en los defectos de fabricación, con el fin de ajustar sus parámetros de producción y reducir el porcentaje de piezas defectuosas.

## 2. Variables generadas (dataset sintético, n = 500)

| Variable | Tipo |
|---|---|
| Temperatura de Producción | Numérica |
| Presión de Máquina | Numérica |
| Tiempo de Operación | Numérica |
| Velocidad de Producción | Numérica |
| Defectuosa | Categórica (Sí/No) |


## 3. Generación de Datos Sintéticos

A continuación se generan los datos sintéticos utilizando `numpy.random`. Se fija una semilla (`np.random.seed(42)`) para garantizar la reproducibilidad de los resultados. Las variables numéricas se generan mediante distribuciones estadísticas conocidas (normal, uniforme, exponencial, poisson, etc.) que aproximan el comportamiento real del fenómeno estudiado, y la variable categórica binaria se construye a partir de una probabilidad condicionada a otras variables, de modo que el dataset conserve relaciones realistas y útiles para el análisis exploratorio.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)


In [ ]:
n = 500

temperatura = np.random.normal(180, 15, n)          # temperatura óptima ~180°C
presion = np.random.normal(50, 8, n)                 # presión óptima ~50 psi
tiempo_operacion = np.random.uniform(1, 12, n)        # horas
velocidad = np.random.normal(100, 20, n)              # unidades/hora

# La probabilidad de defecto aumenta cuando la temperatura o la velocidad se alejan del óptimo
desv_temp = np.abs(temperatura - 180) / 15
desv_vel = np.abs(velocidad - 100) / 20
prob_defecto = 0.03 + 0.07 * desv_temp + 0.05 * desv_vel
prob_defecto = np.clip(prob_defecto, 0, 0.85)
defectuosa = np.where(np.random.rand(n) < prob_defecto, "Sí", "No")

piezas = pd.DataFrame({
    "Temperatura": temperatura.round(2),
    "Presion": presion.round(2),
    "Tiempo_Operacion": tiempo_operacion.round(2),
    "Velocidad": velocidad.round(2),
    "Defectuosa": defectuosa
})
piezas.head()


## 4. Análisis Estadístico (NumPy)

In [ ]:
numeric_cols = ['Temperatura', 'Presion', 'Tiempo_Operacion', 'Velocidad']

resumen = pd.DataFrame({
    "Media": piezas[numeric_cols].mean(),
    "Mediana": piezas[numeric_cols].median(),
    "Moda": piezas[numeric_cols].apply(lambda x: stats.mode(x, keepdims=True).mode[0]),
    "Desviación estándar": piezas[numeric_cols].std(),
    "Varianza": piezas[numeric_cols].var(),
    "Mínimo": piezas[numeric_cols].min(),
    "Máximo": piezas[numeric_cols].max(),
})
resumen.round(2)


**Interpretación:** La media y la mediana de la temperatura cercanas a 180°C confirman que el proceso opera, en promedio, alrededor del punto óptimo de fabricación. La varianza y la desviación estándar muestran qué tanto se aleja el proceso de ese punto óptimo en cada lote, lo cual está directamente relacionado con la aparición de defectos.

## 5. Visualización de Datos (Matplotlib y Seaborn)

### 5.1 Histograma de temperatura

In [ ]:
plt.figure()
plt.hist(piezas["Temperatura"], bins=20, color="#DD8452", edgecolor="white")
plt.title("Distribución de la Temperatura de Producción")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Frecuencia")
plt.show()


**Interpretación:** La temperatura sigue una distribución aproximadamente normal centrada en 180°C, con variaciones propias del proceso productivo.

### 5.2 Histograma de presión

In [ ]:
plt.figure()
plt.hist(piezas["Presion"], bins=20, color="#55A868", edgecolor="white")
plt.title("Distribución de la Presión de Máquina")
plt.xlabel("Presión (psi)")
plt.ylabel("Frecuencia")
plt.show()


**Interpretación:** La presión también se concentra alrededor de un valor central, lo que indica un proceso relativamente estable, aunque con dispersión suficiente para generar variaciones en la calidad final.

### 5.3 Gráfico de barras de piezas defectuosas

In [ ]:
conteo_defectos = piezas["Defectuosa"].value_counts()
plt.figure()
plt.bar(conteo_defectos.index, conteo_defectos.values, color=["#55A868", "#C44E52"])
plt.title("Cantidad de Piezas Defectuosas vs. Sanas")
plt.xlabel("¿Defectuosa?")
plt.ylabel("Cantidad de piezas")
plt.show()


**Interpretación:** Este gráfico evidencia la magnitud del problema de calidad: la proporción de piezas defectuosas frente al total de piezas producidas en el periodo analizado.

### 5.4 Matriz de correlación (Heatmap)

In [ ]:
plt.figure()
correlacion = piezas[["Temperatura", "Presion", "Tiempo_Operacion", "Velocidad"]].corr()
sns.heatmap(correlacion, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de Correlación entre Variables del Proceso")
plt.show()


**Interpretación:** El heatmap permite ver si existen relaciones lineales fuertes entre las variables del proceso. En general, temperatura, presión, tiempo y velocidad son controladas de forma relativamente independiente, por lo que las correlaciones entre ellas suelen ser bajas.

### 5.5 Boxplot de temperatura

In [ ]:
plt.figure()
sns.boxplot(x=piezas["Temperatura"], color="#8172B2")
plt.title("Boxplot de Temperatura de Producción")
plt.show()


**Interpretación:** El boxplot permite identificar lotes con temperaturas atípicas (muy altas o muy bajas), que son los principales sospechosos de generar piezas defectuosas.

### 5.6 Pairplot de variables

In [ ]:
sns.pairplot(piezas, vars=["Temperatura", "Presion", "Tiempo_Operacion", "Velocidad"], hue="Defectuosa")
plt.show()


**Interpretación:** El pairplot diferenciado por color permite observar visualmente si las piezas defectuosas se concentran en zonas específicas del espacio de variables, típicamente en los extremos de temperatura y velocidad.

### 5.7 Boxplot comparativo: temperatura según defecto

In [ ]:
plt.figure()
sns.boxplot(x="Defectuosa", y="Temperatura", data=piezas, palette=["#55A868", "#C44E52"])
plt.title("Temperatura según condición de la pieza")
plt.show()


**Interpretación:** Si las piezas defectuosas presentan una mayor dispersión o un promedio alejado de 180°C respecto a las piezas sanas, esto confirma que la temperatura es un factor relevante en la aparición de defectos.

## 6. Análisis Exploratorio

In [ ]:
desv_temp_grupo = piezas.assign(Desviacion_Temp=np.abs(piezas["Temperatura"] - 180)).groupby("Defectuosa")["Desviacion_Temp"].mean()
print("Desviación promedio respecto a 180°C, por condición de la pieza:")
print(desv_temp_grupo.round(2))


**Hallazgos:**
- Las piezas defectuosas muestran, en promedio, una mayor desviación de la temperatura respecto al valor óptimo (180°C) que las piezas sanas.
- La velocidad de producción extrema (muy alta o muy baja) también se asocia con mayor proporción de defectos.
- La presión y el tiempo de operación, dentro de los rangos generados, no muestran una asociación tan marcada con los defectos.

**Relaciones entre variables:** la principal relación encontrada es entre la desviación de temperatura/velocidad respecto a sus valores óptimos y la probabilidad de defecto, más que relaciones lineales directas entre las variables del proceso entre sí.

**Variable más relevante:** la Temperatura de Producción, seguida de la Velocidad de Producción.

## 7. Conclusiones

1. Las piezas defectuosas tienden a producirse cuando la temperatura se aleja significativamente del valor óptimo de operación.
2. La velocidad de producción extrema también está asociada con un mayor porcentaje de defectos.
3. La presión de máquina, dentro del rango analizado, no muestra una relación tan clara con la aparición de defectos.
4. Existen valores atípicos de temperatura que coinciden con los lotes de mayor tasa de defectos, lo que sugiere fallos puntuales del proceso o de calibración.
5. El control simultáneo de temperatura y velocidad parece ser más efectivo para reducir defectos que el control aislado de una sola variable.

## 8. Recomendaciones Empresariales

1. Implementar un sistema de monitoreo en tiempo real de temperatura y velocidad, con alertas cuando se alejen del rango óptimo.
2. Revisar y calibrar periódicamente las máquinas asociadas a los lotes con temperaturas atípicas detectadas en el boxplot.
3. Establecer rangos de control estadístico de proceso (SPC) para temperatura y velocidad, deteniendo la producción cuando se excedan los límites definidos.

## 9. Reflexión de IA

**¿Cómo podrían utilizarse técnicas de Machine Learning o Inteligencia Artificial para automatizar la toma de decisiones en este caso?**

podriamos entrenar a los modelos de Inteligencia Artificial dependiendo de que metodo de clasificación
(árboles de decisión, random forest o redes neuronales) nos interese elegir, y que estos predigan la probabilidad
de que una pieza resulte defectuosa usando los datos de temperatura, presión, tiempo y velocidad
registradas durante su fabricación. Estos modelos podrian activar alertas automaticas o incluso
detener la línea de producción casi en tiempo real cuando se detecten que los parámetros
asociados a el alto riesgo de que salgan defectuosas se cumplieron, esto reduciria el desperdicio de materiales y
tambien mejoraria la eficiencia del control de calidad.